In [ ]:
%pip install -q chromadb sentence-transformers langchain langchain-community pdfplumber tiktoken pandas numpylangchain_text_splitters

In [ ]:
%pip install langchain_text_splitters

In [ ]:
import os, re, time, warnings
import numpy as np
import pandas as pd
import pdfplumber, tiktoken, chromadb
from pathlib import Path
from typing import List, Dict, Tuple
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
warnings.filterwarnings("ignore")

# ── config ────────────────────────────────────────────────────
PDF_PATH          = "hadith.pdf"      # put PDF in same folder as notebook
EMBED_MODEL       = "all-MiniLM-L6-v2"  # 384-dim, ~80MB, fast & good
CHROMA_DIR        = "./chroma_store"   # local folder, survives restarts
TOP_K             = 3
CHUNK_SIZE_TOKENS = 256
OVERLAP_TOKENS    = 50
SEMANTIC_THRESH   = 0.3              # cosine drop that triggers a split

print(f"PDF: {PDF_PATH} | Embed: {EMBED_MODEL} | ChromaDB: {CHROMA_DIR}")

In [ ]:
def extract_bukhari_pdf(path: str) -> Tuple[str, List[Dict]]:
    """
    Returns:
        full_text : whole book as one string
        hadiths   : list of {number, narrator, text, page} dicts
    """
    pages_text = []
    with pdfplumber.open(path) as pdf:
        total = len(pdf.pages)
        print(f"  {total} pages found…")
        for i, page in enumerate(pdf.pages):
            if i < 6:  # skip cover + intro pages
                continue
            text = page.extract_text()
            if not text: continue
            # strip running headers/footers (lines like "Volume 1 - 8 / 1700")
            lines = text.split("\n")
            lines = [l for l in lines
                     if not re.match(r"Volume \d+ - \d+ / \d+", l.strip())
                     and not re.match(r"SAHIH BUKHARI VOLUME", l.strip())]
            pages_text.append("\n".join(lines).strip())

    full_text = "\n\n".join(pages_text)

    # parse individual hadiths using the "Volume N, Book N, Number N:" pattern
    hadith_pattern = re.compile(
        r"(Volume\s+\d+,\s+Book\s+\d+,\s+Number\s+\d+:)",
        re.IGNORECASE
    )
    parts = hadith_pattern.split(full_text)
    hadiths = []
    for j in range(1, len(parts) - 1, 2):
        header = parts[j].strip()
        body   = parts[j + 1].strip()
        # extract narrator line ("Narrated X:")
        narrator_match = re.match(r"(Narrated [^:]+:)", body)
        narrator = narrator_match.group(1) if narrator_match else ""
        hadiths.append({
            "header"  : header,
            "narrator": narrator,
            "text"    : (header + "\n" + body).strip(),
        })

    print(f"  Extracted {len(hadiths)} individual hadiths")
    print(f"  Total characters: {len(full_text):,}")
    return full_text, hadiths

full_text, hadiths = extract_bukhari_pdf(PDF_PATH)
print(f"\nFirst hadith header: {hadiths[0]['header']}")
print(f"First narrator: {hadiths[0]['narrator']}")
print(f"Preview: {hadiths[0]['text'][:300]}…")


In [3]:
enc = tiktoken.get_encoding("cl100k_base")  # same tokenizer as GPT-4

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def chunk_stats(chunks: List[str], label: str) -> Dict:
    tc = [count_tokens(c) for c in chunks]
    return {
        "strategy"     : label,
        "n_chunks"     : len(chunks),
        "min_tokens"   : int(np.min(tc)),
        "max_tokens"   : int(np.max(tc)),
        "mean_tokens"  : round(float(np.mean(tc)), 1),
        "median_tokens": int(np.median(tc)),
    }

total_tokens = count_tokens(full_text)
print(f"Total tokens: {total_tokens:,}")
print(f"At chunk_size={CHUNK_SIZE_TOKENS} → ~{total_tokens // CHUNK_SIZE_TOKENS} chunks")
print(f"Total hadiths parsed: {len(hadiths)} → these become chunks in strategy 4b")

Total tokens: 958,252
At chunk_size=256 → ~3743 chunks
Total hadiths parsed: 6650 → these become chunks in strategy 4b


In [4]:
# ── 1: fixed-size, no overlap ─────────────────────────────────
def chunk_fixed(text: str) -> List[str]:
    tokens = enc.encode(text)
    chunks = []
    for i in range(0, len(tokens), CHUNK_SIZE_TOKENS):
        chunks.append(enc.decode(tokens[i:i + CHUNK_SIZE_TOKENS]))
    return [c for c in chunks if len(c.strip()) > 20]

# ── 2: fixed-size + overlap ───────────────────────────────────
def chunk_overlap(text: str) -> List[str]:
    tokens = enc.encode(text)
    step = CHUNK_SIZE_TOKENS - OVERLAP_TOKENS
    chunks = []
    for i in range(0, len(tokens), step):
        sl = tokens[i:i + CHUNK_SIZE_TOKENS]
        if len(sl) < 10: break
        chunks.append(enc.decode(sl))
    return chunks

# ── 3: recursive character split ──────────────────────────────
def chunk_recursive(text: str) -> List[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_TOKENS * 4,    # ~chars (1 token ≈ 4 chars)
        chunk_overlap=OVERLAP_TOKENS * 4,
        separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
    )
    docs = splitter.create_documents([text])
    return [d.page_content for d in docs if len(d.page_content.strip()) > 20]

# ── 4a: semantic split ────────────────────────────────────────
def chunk_semantic(text: str, model: SentenceTransformer) -> List[str]:
    sentences = [s.strip() for s in
                 re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 10]
    if len(sentences) < 3: return [text]
    print(f"  Encoding {len(sentences)} sentences…", end=" ")
    embs = model.encode(sentences, batch_size=64, show_progress_bar=False)
    def cos(a, b): return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-9)
    sims = [cos(embs[i], embs[i+1]) for i in range(len(embs)-1)]
    splits = [0] + [i+1 for i,s in enumerate(sims) if s < (1-SEMANTIC_THRESH)] + [len(sentences)]
    chunks = []
    for st, en in zip(splits[:-1], splits[1:]):
        chunk = " ".join(sentences[st:en])
        tok = count_tokens(chunk)
        if tok < 50 and chunks: chunks[-1] += " " + chunk
        elif tok > CHUNK_SIZE_TOKENS * 2: chunks.extend(chunk_recursive(chunk))
        else: chunks.append(chunk)
    print("done")
    return [c for c in chunks if len(c.strip()) > 20]

# ── 4b: hadith-aware (Bukhari-specific) ───────────────────────
# Each parsed hadith is one chunk. Naturally aligned with the source.
# Long hadiths are split with recursive fallback; short ones merged.
def chunk_hadith_aware(hadiths: List[Dict]) -> List[str]:
    chunks = []
    for h in hadiths:
        tok = count_tokens(h["text"])
        if tok > CHUNK_SIZE_TOKENS * 2:          # very long hadith → split
            chunks.extend(chunk_recursive(h["text"]))
        elif tok < 30 and chunks:                # tiny hadith → merge into last
            chunks[-1] += "\n\n" + h["text"]
        else:
            chunks.append(h["text"])
    return chunks

print("✓ all 5 chunking functions defined")

✓ all 5 chunking functions defined


In [11]:
# load embed model once — reused by semantic chunking and ChromaDB
print(f"Loading {EMBED_MODEL}…")
embed_model = SentenceTransformer(EMBED_MODEL)
print("✓ model loaded\n")

print("[1/5] fixed, no overlap")
chunks_fixed    = chunk_fixed(full_text)

print("[2/5] fixed + overlap")
chunks_overlap  = chunk_overlap(full_text)

print("[3/5] recursive")
chunks_rec      = chunk_recursive(full_text)

print("[4a/5] semantic")
chunks_sem      = chunk_semantic(full_text, embed_model)

print("[4b/5] hadith-aware")
chunks_hadith   = chunk_hadith_aware(hadiths)

all_strats = [
    ("fixed_no_overlap", chunks_fixed),
    ("fixed_overlap",    chunks_overlap),
    ("recursive",        chunks_rec),
    ("semantic",         chunks_sem),
    ("hadith_aware",     chunks_hadith),
]

stats = [chunk_stats(chunks, label) for label, chunks in all_strats]
df_stats = pd.DataFrame(stats).set_index("strategy")
print("\n── chunk statistics ─────────────────────────")
display(df_stats)

# eyeball first chunk per strategy to spot boundary problems
for label, chunks in all_strats:
    print(f"\n▸ {label} | first chunk ({count_tokens(chunks[0])} tokens)")
    print(chunks[0][:300])

Loading all-MiniLM-L6-v2…


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1677.77it/s]


✓ model loaded

[1/5] fixed, no overlap
[2/5] fixed + overlap
[3/5] recursive
[4a/5] semantic
  Encoding 18676 sentences… done
[4b/5] hadith-aware

── chunk statistics ─────────────────────────


,n_chunks,min_tokens,max_tokens,mean_tokens,median_tokens
strategy,,,,,
fixed_no_overlap,3744,44,256,255.9,256
fixed_overlap,4652,146,256,256.0,256
recursive,4832,26,356,225.5,251
semantic,6719,50,741,142.4,121
hadith_aware,7128,13,516,136.8,109



▸ fixed_no_overlap | first chunk (256 tokens)
Book 1: Revelation
Volume 1, Book 1, Number 1:
Narrated 'Umar bin Al-Khattab:
I heard Allah's Apostle saying, "The reward of deeds depends upon the intentions and every person
will get the reward according to what he has intended. So whoever emigrated for worldly benefits or
for a woman to marry, hi

▸ fixed_overlap | first chunk (256 tokens)
Book 1: Revelation
Volume 1, Book 1, Number 1:
Narrated 'Umar bin Al-Khattab:
I heard Allah's Apostle saying, "The reward of deeds depends upon the intentions and every person
will get the reward according to what he has intended. So whoever emigrated for worldly benefits or
for a woman to marry, hi

▸ recursive | first chunk (245 tokens)
Book 1: Revelation
Volume 1, Book 1, Number 1:
Narrated 'Umar bin Al-Khattab:
I heard Allah's Apostle saying, "The reward of deeds depends upon the intentions and every person
will get the reward according to what he has intended. So whoever emigrated for worldly ben

In [5]:
# ── Cell 5 shortcut — kernel restarted, ChromaDB already indexed ──
# skip rechunking — just reload embed model and define dummy lists
# we only need these variables to exist for Cell 9+ eval code to run

print(f"Loading {EMBED_MODEL}...")
embed_model = SentenceTransformer(EMBED_MODEL)
print("✓ model loaded")

# minimal stubs — just enough for eval and Cell 11b MMR to work
# we won't rechunk, we'll load directly from ChromaDB
all_strats = [
    ("fixed_no_overlap", []),
    ("fixed_overlap",    []),
    ("recursive",        []),
    ("semantic",         []),
    ("hadith_aware",     []),
]
print("✓ stubs ready — collections will load from disk in Cell 6")

Loading all-MiniLM-L6-v2...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1775.33it/s]


✓ model loaded
✓ stubs ready — collections will load from disk in Cell 6


In [12]:
# ── persistent client — folder created automatically ──────────
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
# └── chroma_store/ will appear in your project folder
#     delete it to start fresh on next run

# ── wrap sentence-transformers as a ChromaDB EmbeddingFunction ─
#    ChromaDB needs this interface; we reuse the already-loaded model
class LocalEF(chromadb.EmbeddingFunction):
    def __init__(self, model): self.model = model
    def __call__(self, input):
        return self.model.encode(input, batch_size=64,
                                   show_progress_bar=False).tolist()

local_ef = LocalEF(embed_model)
dim = embed_model.get_sentence_embedding_dimension()
print(f"✓ ChromaDB ready at {CHROMA_DIR}")
print(f"✓ Embedding fn ready | model={EMBED_MODEL} | dim={dim}")

✓ ChromaDB ready at ./chroma_store
✓ Embedding fn ready | model=all-MiniLM-L6-v2 | dim=384


In [6]:
# reload ChromaDB from disk — all 5 collections already there
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

class LocalEF(chromadb.EmbeddingFunction):
    def __init__(self, model): self.model = model
    def __call__(self, input):
        return self.model.encode(input, batch_size=64,
                                 show_progress_bar=False).tolist()

local_ef = LocalEF(embed_model)

# reload all 5 collections from disk
collections = {}
for label, _ in all_strats:
    collections[label] = chroma_client.get_collection(
        name=label,
        embedding_function=local_ef
    )
    count = collections[label].count()
    print(f"✓ loaded '{label}' — {count} chunks")

print("\n✓ all collections restored from disk — no reindexing needed")

✓ loaded 'fixed_no_overlap' — 3744 chunks
✓ loaded 'fixed_overlap' — 4652 chunks
✓ loaded 'recursive' — 4832 chunks
✓ loaded 'semantic' — 6719 chunks
✓ loaded 'hadith_aware' — 7128 chunks

✓ all collections restored from disk — no reindexing needed


In [13]:
#Each strategy gets its own collection (ChromaDB's equivalent of a Pinecone index).
#We store hadith header metadata alongside each chunk — useful for debugging which hadith a retrieved chunk came from.
def index_chunks(client, name: str, chunks: List[str], ef) -> chromadb.Collection:
    try: client.delete_collection(name)   # clean rerun
    except: pass

    col = client.create_collection(
        name=name,
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"},  # cosine similarity
    )
    # batch in 500s — safe for large Hadith corpus
    for start in range(0, len(chunks), 500):
        batch = chunks[start:start + 500]
        col.add(
            documents=batch,
            ids=[f"{name}_{start+i}" for i in range(len(batch))],
            metadatas=[{
                "chunk_index": start + i,
                "token_count": count_tokens(c),
                # detect if chunk contains a hadith header for traceability
                "has_header"  : bool(re.search(r"Volume \d+, Book \d+", c)),
            } for i, c in enumerate(batch)]
        )
    return col

collections = {}
for label, chunks in all_strats:
    print(f"Indexing '{label}' ({len(chunks)} chunks)…", end=" ")
    t0 = time.time()
    collections[label] = index_chunks(chroma_client, label, chunks, local_ef)
    print(f"done in {time.time()-t0:.1f}s")

print("\n✓ all collections in ChromaDB at ./chroma_store")
print(f"collections: {list(collections.keys())}")

Indexing 'fixed_no_overlap' (3744 chunks)… done in 277.4s
done in 491.0sd_overlap' (4652 chunks)… 
done in 362.5srsive' (4832 chunks)… 
done in 340.4sntic' (6719 chunks)… 
done in 339.2sth_aware' (7128 chunks)… 

✓ all collections in ChromaDB at ./chroma_store
collections: ['fixed_no_overlap', 'fixed_overlap', 'recursive', 'semantic', 'hadith_aware']


Queries are grounded in actual Bukhari content. The answer_keywords are distinctive phrases from the text — not common words that appear everywhere. For best results, open your PDF and verify each keyword actually appears.
The hadith_ref field is for your notes — it helps you verify the keyword is actually in the PDF when debugging misse

In [7]:
EVAL_SET = [
    {"query": "What did the Prophet say about intentions behind actions?",
     "answer_keywords": ["reward of deeds depends upon the intentions"],
     "hadith_ref": "Vol 1 Book 1 #1 — classic opening hadith"},

    {"query": "How did the revelation come to the Prophet Muhammad?",
     "answer_keywords": ["I do not know how to read."],
     "hadith_ref": "Vol 1 Book 1 — first revelation hadiths"},

    {"query": "What is the generosity of the Prophet in Ramadan?",
     "answer_keywords": [". Gabriel used to meet him every night of Ramadan to teach him the Quran"],
     "hadith_ref": "Vol 1 Book 1 #5"},

    {"query": "What are the five pillars of Islam?",
     "answer_keywords": ["prayer", "fasting", "zakat", "hajj", "testimony"],
     "hadith_ref": "Vol 1 Book 2 — belief"},

    {"query": "How should ablution (wudu) be performed?",
     "answer_keywords": ["wudu", "ablution", "wash his face", "forearms"],
     "hadith_ref": "Vol 1 Book 4"},

    {"query": "What is the ruling on ghusl after intercourse?",
     "answer_keywords": ["ghusl", "bathing", "major impurity", "janaba"],
     "hadith_ref": "Vol 1 Book 5"},

    {"query": "What did the Prophet say about seeking knowledge?",
     "answer_keywords": ["knowledge", "learn", "scholars", "teach"],
     "hadith_ref": "Vol 1 Book 3 — Knowledge"},

    {"query": "What is the adhan (call to prayer) wording?",
     "answer_keywords": ["adhan", "Allahu Akbar", "hayya", "muezzin"],
     "hadith_ref": "Vol 1 Book 11"},

    {"query": "What would happened to the least punished person of Hell? ",
     "answer_keywords": ["man under whose arch of the feet two smoldering embers will be placed, because of which his brain"],
     "hadith_ref": "Volume 8 Book 76"},

    {"query": "What is tayammum and when is it used?",
     "answer_keywords": ["tayammum", "dust", "sand", "no water"],
     "hadith_ref": "Vol 1 Book 7"},

    {"query": "What is the virtue of the Friday prayer?",
     "answer_keywords": ["Friday", "Jumu'ah", "khutbah", "sermon"],
     "hadith_ref": "Vol 2 Book 13"},

    {"query": "How did Abu Sufyan describe the Prophet to Heraclius?",
     "answer_keywords": ["Abu Sufyan", "Heraclius", "Caesar"],
     "hadith_ref": "Vol 1 Book 1 #6 — long narrative hadith"},
]

print(f"✓ {len(EVAL_SET)} eval queries ready")
for i, q in enumerate(EVAL_SET, 1):
    print(f"  {i:2}. {q['query'][:65]}")

✓ 12 eval queries ready
   1. What did the Prophet say about intentions behind actions?
   2. How did the revelation come to the Prophet Muhammad?
   3. What is the generosity of the Prophet in Ramadan?
   4. What are the five pillars of Islam?
   5. How should ablution (wudu) be performed?
   6. What is the ruling on ghusl after intercourse?
   7. What did the Prophet say about seeking knowledge?
   8. What is the adhan (call to prayer) wording?
   9. What would happened to the least punished person of Hell? 
  10. What is tayammum and when is it used?
  11. What is the virtue of the Friday prayer?
  12. How did Abu Sufyan describe the Prophet to Heraclius?


hit@k: was the answer in any of the top-k chunks? MRR: if yes, was it rank 1 (score=1.0) or rank 3 (score=0.33)? 
avg_ret_tokens: bigger chunks pass more context to the LLM but also more noise.

In [8]:
def evaluate(collection, eval_set, top_k=TOP_K):
    results = []
    for item in eval_set:
        kws  = [k.lower() for k in item["answer_keywords"]]
        resp = collection.query(
            query_texts=[item["query"]],
            n_results=top_k,
            include=["documents", "distances", "metadatas"],
        )
        chunks = resp["documents"][0]
        metas  = resp["metadatas"][0]
        hit_rank = None
        for rank, chunk in enumerate(chunks, 1):
            if any(kw in chunk.lower() for kw in kws):
                hit_rank = rank; break
        results.append({
            "query"      : item["query"][:48],
            "hit"        : hit_rank is not None,
            "hit_rank"   : hit_rank,
            "rr"         : round(1.0/hit_rank, 3) if hit_rank else 0.0,
            "avg_tokens" : round(np.mean([m["token_count"] for m in metas]), 1),
        })
    hit_at_k = sum(r["hit"] for r in results) / len(results)
    mrr      = float(np.mean([r["rr"] for r in results]))
    return results, {f"hit@{top_k}": round(hit_at_k,3), "MRR": round(mrr,3),
                     "avg_tokens": round(float(np.mean([r["avg_tokens"] for r in results])),1)}

all_results, all_metrics = {}, {}
for label, _ in all_strats:
    print(f"Evaluating '{label}'…", end=" ")
    t0 = time.time()
    all_results[label], all_metrics[label] = evaluate(collections[label], EVAL_SET)
    m = all_metrics[label]
    print(f"done in {time.time()-t0:.1f}s | hit@{TOP_K}={m[f'hit@{TOP_K}']} MRR={m['MRR']}")

print("✓ evaluation complete")

Evaluating 'fixed_no_overlap'… done in 0.6s | hit@3=0.583 MRR=0.542
done in 0.3s | hit@3=0.583 MRR=0.583
done in 0.3s | hit@3=0.667 MRR=0.583
done in 0.3s | hit@3=0.583 MRR=0.542
done in 0.3s | hit@3=0.667 MRR=0.583
✓ evaluation complete


In [ ]:
# summary: one row per strategy
rows = []
for label, _ in all_strats:
    st = df_stats.loc[label].to_dict()
    mt = all_metrics[label]
    rows.append({"strategy": label, "n_chunks": st["n_chunks"],
                 "median_tok": st["median_tokens"],
                 f"hit@{TOP_K}": mt[f"hit@{TOP_K}"], "MRR": mt["MRR"],
                 "avg_ret_tok": mt["avg_tokens"]})

df_out = pd.DataFrame(rows).set_index("strategy")
display(df_out)

best_hit = df_out[f"hit@{TOP_K}"].idxmax()
best_mrr = df_out["MRR"].idxmax()
print(f"\n best hit@{TOP_K} : {best_hit}")
print(f" best MRR     : {best_mrr}")

# per-query hit grid
grid = {}
for label, _ in all_strats:
    grid[label] = ["✓" if r["hit"] else "✗" for r in all_results[label]]
df_grid = pd.DataFrame(grid, index=[q["query"][:40] for q in EVAL_SET])
print("\nPer-query hit grid (✓ = keyword found in top-k):")
display(df_grid)

# flag universal misses
misses = [EVAL_SET[i]["query"][:50] for i in range(len(EVAL_SET))
          if all(not all_results[l][i]["hit"] for l, _ in all_strats)]
if misses:
    print("\n⚠ missed by ALL strategies — check keywords or PDF content:")
    for m in misses: print(f"   - {m}")

In [11]:
INSPECT_QUERY = "How did the revelation come to Prophet Muhammad?"

print(f"Query: {INSPECT_QUERY}\n" + "="*60)
for label, _ in all_strats:
    resp = collections[label].query(
        query_texts=[INSPECT_QUERY], n_results=TOP_K,
        include=["documents", "distances", "metadatas"],
    )
    print(f"\n▸ {label}")
    for rank, (chunk, dist, meta) in enumerate(
            zip(resp["documents"][0], resp["distances"][0],
                resp["metadatas"][0]), 1):
        tok = meta["token_count"]
        hdr = "[has header]" if meta["has_header"] else ""
        print(f"  rank {rank} | dist={dist:.3f} | {tok} tokens {hdr}")
        print(f"  {chunk[:250].strip()}…\n")

Query: How did the revelation come to Prophet Muhammad?

▸ fixed_no_overlap
  rank 1 | dist=0.396 | 256 tokens [has header]
  was walking, all of a sudden I heard a voice from the sky.
I looked up and saw the same angel who had visited me at the cave of Hira' sitting on a chair
between the sky and the earth. I got afraid of him and came back home and said, 'Wrap me (in
bla…

  rank 2 | dist=0.442 | 256 tokens [has header]
  it has
been revealed to you?" He said, "I like to hear it from another person." So I recited Surat An-Nisa (The
Women) till I reached the Verse: 'How (will it be) then when We bring from each nation a witness,
and We bring you (O Muhammad) as a witn…

  rank 3 | dist=0.457 | 256 tokens [has header]
  Allah and Muhammad is Allah's Apostle) and the people attended to Abu Bakr and left 'Umar. Abu
Bakr said, "Amma ba'du, whoever amongst you worshipped Muhammad, then Muhammad is dead,
but whoever worshipped Allah, Allah is alive and will never die. Al…


▸ fixed_overlap


In [12]:
# ── Cell 11b: MMR retrieval vs standard top-k ─────────────────────────────────
# WHY: standard top-k often returns redundant chunks (same hadith, different narrators
# saying almost identical things). MMR trades a little relevance for more diversity —
# so your 3 slots aren't wasted on near-duplicate content.
#
# HOW MMR WORKS:
# 1. fetch a larger candidate pool (fetch_k = 15-20)
# 2. pick the first chunk normally (most similar to query)
# 3. for each remaining slot: score = λ * similarity_to_query
#                                   - (1-λ) * max_similarity_to_already_picked
#    → rewards chunks that are relevant BUT different from what's already selected
# 4. λ (lambda) controls the tradeoff: 1.0 = pure similarity, 0.0 = pure diversity
#
# EXPECTED OUTPUT:
# you'll see two retrieval result blocks side by side for the same query:
#   standard top-k  → likely 2-3 chunks from same narrator / same wording
#   MMR             → chunks from different narrators covering the same topic
# the MMR distances will be slightly higher (less similar) but content more varied
# this is the tradeoff: a bit less precision, a lot more coverage

import numpy as np

def mmr_retrieve(
    collection,
    query: str,
    k: int = TOP_K,
    fetch_k: int = 15,       # candidate pool — fetch more, then re-rank
    lambda_val: float = 0.6, # 0.6 = balanced. closer to 1 = more similarity,
                             #                 closer to 0 = more diversity
    embed_model=embed_model,
):
    """
    Manual MMR over ChromaDB results.
    ChromaDB doesn't have MMR built-in so we:
      1. fetch fetch_k candidates with standard similarity search
      2. re-rank them using the MMR scoring formula
      3. return top k after re-ranking
    """
    # step 1 — get candidate pool (more than we need)
    resp = collection.query(
        query_texts=[query],
        n_results=fetch_k,
        include=["documents", "distances", "embeddings"],
    )
    candidates     = resp["documents"][0]       # list of fetch_k chunk strings
    # ChromaDB returns cosine DISTANCE (0=identical, 2=opposite)
    # convert to similarity: sim = 1 - distance
    query_sims     = [1 - d for d in resp["distances"][0]]
    candidate_embs = np.array(resp["embeddings"][0])  # shape (fetch_k, dim)

    def cosine_sim(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

    # step 2 — MMR greedy selection loop
    selected_indices = []
    remaining        = list(range(len(candidates)))

    while len(selected_indices) < k and remaining:
        mmr_scores = {}
        for i in remaining:
            # relevance term: how similar is this chunk to the query?
            relevance = query_sims[i]

            # redundancy term: how similar is this chunk to already-selected chunks?
            # we want this to be HIGH so we can PENALISE it
            if selected_indices:
                redundancy = max(
                    cosine_sim(candidate_embs[i], candidate_embs[j])
                    for j in selected_indices
                )
            else:
                redundancy = 0.0  # nothing selected yet — no redundancy to penalise

            # MMR score: balance relevance vs redundancy
            mmr_scores[i] = lambda_val * relevance - (1 - lambda_val) * redundancy

        # pick the candidate with the highest MMR score
        best = max(mmr_scores, key=mmr_scores.get)
        selected_indices.append(best)
        remaining.remove(best)

    return [candidates[i] for i in selected_indices], \
           [query_sims[i]  for i in selected_indices]


# ── run comparison on the same query ──────────────────────────────────────────
# change this to any query — works best on topics with multiple narrators
COMPARE_QUERY = "What did the Prophet say about intentions?"

# pick the best-performing collection from your Cell 10 results
# (swap "hadith_aware" to whichever won your eval)
COMPARE_COLLECTION = collections["hadith_aware"]

print(f"Query: {COMPARE_QUERY}")
print("=" * 65)

# ── standard top-k ────────────────────────────────────────────────────────────
print(f"\n▸ STANDARD top-{TOP_K} (pure similarity)")
print("  expect: highest similarity scores, possibly redundant narrators\n")
std_resp = COMPARE_COLLECTION.query(
    query_texts=[COMPARE_QUERY],
    n_results=TOP_K,
    include=["documents", "distances"],
)
for rank, (chunk, dist) in enumerate(
        zip(std_resp["documents"][0], std_resp["distances"][0]), 1):
    sim = round(1 - dist, 3)   # convert distance → similarity for readability
    tok = count_tokens(chunk)
    print(f"  rank {rank} | sim={sim} | {tok} tokens")
    print(f"  {chunk[:220].strip()}…\n")

# ── MMR retrieval ─────────────────────────────────────────────────────────────
print(f"\n▸ MMR top-{TOP_K} (lambda=0.6, fetch pool=15)")
print("  expect: slightly lower sim scores, but different narrators / angles\n")
mmr_chunks, mmr_sims = mmr_retrieve(
    COMPARE_COLLECTION, COMPARE_QUERY,
    k=TOP_K, fetch_k=15, lambda_val=0.6,
)
for rank, (chunk, sim) in enumerate(zip(mmr_chunks, mmr_sims), 1):
    tok = count_tokens(chunk)
    print(f"  rank {rank} | sim={round(sim,3)} | {tok} tokens")
    print(f"  {chunk[:220].strip()}…\n")

# ── diversity check ───────────────────────────────────────────────────────────
# measure pairwise similarity between retrieved chunks
# low avg = diverse results (good), high avg = redundant results (bad)
print("── diversity check (lower = more diverse) ────────────────")
def avg_pairwise_sim(chunks, model):
    if len(chunks) < 2: return 1.0
    embs = model.encode(chunks, show_progress_bar=False)
    sims = []
    for i in range(len(embs)):
        for j in range(i+1, len(embs)):
            sims.append(float(np.dot(embs[i], embs[j]) /
                              (np.linalg.norm(embs[i])*np.linalg.norm(embs[j])+1e-9)))
    return round(float(np.mean(sims)), 3)

std_chunks  = std_resp["documents"][0]
std_div     = avg_pairwise_sim(std_chunks, embed_model)
mmr_div     = avg_pairwise_sim(mmr_chunks, embed_model)

print(f"  standard top-k avg pairwise similarity : {std_div}  ← higher = more redundant")
print(f"  MMR            avg pairwise similarity : {mmr_div}  ← should be lower")
print(f"\n  diversity gain from MMR: {round(std_div - mmr_div, 3)}")
print("  (a positive number means MMR retrieved more varied chunks)")

# ── lambda sensitivity ─────────────────────────────────────────────────────────
# shows how lambda shifts the relevance/diversity tradeoff
print("\n── lambda sensitivity (same query, vary lambda) ──────────")
print("  lambda | avg_sim_to_query | avg_pairwise_sim | interpretation")
for lam in [0.9, 0.7, 0.5, 0.3]:
    mc, ms = mmr_retrieve(COMPARE_COLLECTION, COMPARE_QUERY,
                          k=TOP_K, fetch_k=15, lambda_val=lam)
    avg_rel = round(float(np.mean(ms)), 3)
    avg_div = avg_pairwise_sim(mc, embed_model)
    interp  = "more similarity" if lam >= 0.7 else "more diversity"
    print(f"  {lam}    | {avg_rel}              | {avg_div}            | {interp}")

Query: What did the Prophet say about intentions?

▸ STANDARD top-3 (pure similarity)
  expect: highest similarity scores, possibly redundant narrators

  rank 1 | sim=0.615 | 78 tokens
  and children and those who obeyed me (to help you)?" They said, "Yes." He said, "Well, this man (i.e.
the Prophet) has offered you a reasonable proposal, you'd better accept it and allow me to meet him."
They said, "You…

  rank 2 | sim=0.595 | 203 tokens
  Volume 6, Book 60, Number 245:
Narrated Abdullah:
While I was in the company of the Prophet on a farm and he was reclining on a palm leave stalk,
some Jews passed by. Some of them said to the others. "Ask him (the Prophe…

  rank 3 | sim=0.589 | 129 tokens
  Volume 6, Book 60, Number 145:
Narrated Anas:
The Prophet delivered a sermon the like of which I had never heard before. He said, "If you but
knew what I know then you would have laughed little and wept much." On hearing…


▸ MMR top-3 (lambda=0.6, fetch pool=15)
  expect: slightly lower sim sc

In [13]:
for label, info in {
    "fixed_no_overlap": ("Simple, fast",
                         "Cuts mid-hadith — loses narrator + text continuity",
                         "Baseline only. Never ship this."),
    "fixed_overlap"   : ("Reduces boundary loss",
                         "More chunks, duplicated context retrieved",
                         "Default fallback when time is short"),
    "recursive"       : ("Respects paragraph/sentence boundaries",
                         "Variable chunk size, hard to budget tokens",
                         "Best general-purpose for prose books"),
    "semantic"        : ("Detects topic shifts (new hadith = new topic)",
                         "Slow index build, threshold-sensitive",
                         "Strong when document has clear topic boundaries"),
    "hadith_aware"    : ("Perfectly aligned with document structure",
                         "Requires parsing — only works for structured corpora",
                         "Best for Bukhari/any numbered-section corpus"),
}.items():
    m = all_metrics[label]
    pros, cons, when = info
    print(f"\n{label}")
    print(f"  scores : hit@{TOP_K}={m[f'hit@{TOP_K}']}  MRR={m['MRR']}")
    print(f"  ✓ pros : {pros}")
    print(f"  ✗ cons : {cons}")
    print(f"  ↗ use  : {when}")

print("""
══ PrepSphere retrospective ════════════════════════════════

PrepSphere used OCR → fixed chunking → Pinecone on 800+ pages.
Based on this eval:

1. fixed_no_overlap is almost certainly what was shipped —
   it was losing content at chunk boundaries on every query.

2. The 800-page corpus had natural structure (page numbers,
   section headers). hadith_aware-style parsing would have
   been possible and would have likely given the best results.

3. Without an eval set there was no way to know if changes
   helped or hurt. Any Pinecone experiment was flying blind.

4. ChromaDB local + this eval loop = what dev/staging should
   have looked like. Zero cost. Full observability.

If you rebuild: recursive or hadith_aware chunking +
this eval harness running on every PR.
""")


fixed_no_overlap
  scores : hit@3=0.583  MRR=0.542
  ✓ pros : Simple, fast
  ✗ cons : Cuts mid-hadith — loses narrator + text continuity
  ↗ use  : Baseline only. Never ship this.

fixed_overlap
  scores : hit@3=0.583  MRR=0.583
  ✓ pros : Reduces boundary loss
  ✗ cons : More chunks, duplicated context retrieved
  ↗ use  : Default fallback when time is short

recursive
  scores : hit@3=0.667  MRR=0.583
  ✓ pros : Respects paragraph/sentence boundaries
  ✗ cons : Variable chunk size, hard to budget tokens
  ↗ use  : Best general-purpose for prose books

semantic
  scores : hit@3=0.583  MRR=0.542
  ✓ pros : Detects topic shifts (new hadith = new topic)
  ✗ cons : Slow index build, threshold-sensitive
  ↗ use  : Strong when document has clear topic boundaries

hadith_aware
  scores : hit@3=0.667  MRR=0.583
  ✓ pros : Perfectly aligned with document structure
  ✗ cons : Requires parsing — only works for structured corpora
  ↗ use  : Best for Bukhari/any numbered-section corpus

══ PrepSp

In [ ]:
df_out.to_csv("rag_strategy_comparison.csv")
df_grid.to_csv("rag_per_query_hits.csv")
print("✓ rag_strategy_comparison.csv")
print("✓ rag_per_query_hits.csv")

# show what ChromaDB actually wrote to disk
print("\nChromaDB store contents:")
for f in sorted(Path(CHROMA_DIR).rglob("*")):
    if f.is_file():
        print(f"  {f}  ({f.stat().st_size // 1024} KB)")

# sanity: reload collection from disk and run one query
print("\nReload test — opening persistent client fresh:")
fresh = chromadb.PersistentClient(path=CHROMA_DIR)
col   = fresh.get_collection("hadith_aware", embedding_function=local_ef)
r = col.query(query_texts=["Prophet generosity Ramadan"], n_results=1)
print(f"✓ reloaded. Sample result: {r['documents'][0][0][:150]}…")

In [34]:
import requests
import os
import time


OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
MODEL = "openrouter/free"  # let OpenRouter pick — this actually works

def get_content(raw):
    """handle both normal and reasoning model responses"""
    if "error" in raw:
        print(f"API error: {raw['error']['message'][:80]}")
        return None
    if "choices" not in raw:
        return None
    
    msg = raw["choices"][0]["message"]
    
    # normal model — content field has the answer
    if msg.get("content"):
        return msg["content"]
    
    # reasoning model (like nvidia/nemotron) — answer is in reasoning field
    # extract just the final answer part after the thinking
    if msg.get("reasoning"):
        reasoning = msg["reasoning"]
        # reasoning models think out loud then conclude
        # take the last sentence as the actual answer
        return reasoning
    
    return None

def llm_call(messages, max_tokens=500):
    """single call with openrouter/free router"""
    for attempt in range(3):
        resp = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "HTTP-Referer": "http://localhost:8888",  # helps OpenRouter route better
                "X-Title": "Bukhari RAG"
            },
            json={
                "model": MODEL,
                "messages": messages,
                "max_tokens": max_tokens
            }
        )
        raw = resp.json()
        
        # rate limited
        if raw.get("error", {}).get("code") == 429:
            wait = raw["error"].get("metadata", {}).get("retry_after_seconds", 15)
            print(f"Rate limited — waiting {int(wait)}s (attempt {attempt+1}/3)...")
            time.sleep(wait + 1)
            continue
        
        content = get_content(raw)
        if content:
            routed_to = raw.get("model", "unknown")
            print(f"✓ routed to: {routed_to}")
            return content
        
        print(f"Attempt {attempt+1} failed — retrying...")
        time.sleep(5)
    
    return None


def hyde_retrieve(query, collection, k=3, fetch_k=25):
    print("Generating hypothetical answer...")
    hypothetical = llm_call(
        messages=[{"role": "user", "content": f"""Write 2-3 sentences answering this 
question in hadith style. Use specific physical language. State as fact.

Question: {query}
Answer:"""}],
        max_tokens=200
    )
    
    if not hypothetical:
        print("HyDE failed — using direct query")
        hypothetical = query
    else:
        print(f"Hypothetical: {hypothetical[:200]}\n")
    
    mmr_result = mmr_retrieve(collection, hypothetical, k=k, fetch_k=fetch_k)
    return mmr_result[0], mmr_result[1], hypothetical


def rag_answer(query, collection, top_k=3):
    # step 1 — HyDE retrieval
    chunks, sims, hypothetical = hyde_retrieve(
        query, collection, k=top_k, fetch_k=25
    )
    
    print("\nRetrieved chunks:")
    for i, chunk in enumerate(chunks, 1):
        target = "smoldering" in chunk.lower() or "arch of the feet" in chunk.lower()
        print(f"  rank {i} {'✓ TARGET' if target else '·'}: {chunk[:80]}...")
    print()
    
    context = "\n\n---\n\n".join(chunks)

    print("Generating answer...")
    answer = llm_call(
        messages=[{"role": "user", "content": f"""You are a knowledgeable assistant for Sahih Bukhari.
Use ONLY the provided hadith excerpts to answer the question.
If the answer is not in the excerpts say "I could not find this in the provided hadiths."
Always mention Volume, Book and Number of the hadith you cite.

Hadith excerpts:
{context}

Question: {query}

Answer:"""}],
        max_tokens=500
    )
    return answer if answer else "All models unavailable — try again in a minute."


# ── run ───────────────────────────────────────────────────────
answer = rag_answer(
    "Is it permissible to use kohl for a person whose husband recently died?",
    collections["hadith_aware"]
)
print("=" * 50)
print(answer)

Generating hypothetical answer...
✓ routed to: poolside/laguna-xs.2-20260421:free
Hypothetical: 
Okay, the user wants me to answer whether it's permissible to use kohl after a husband's death, in hadith style. Let me recall the relevant Islamic teachings.

First, I remember that in Islamic juris


Retrieved chunks:
  rank 1 ·: Volume 7, Book 63, Number 252:
Narrated Um Salama:
A woman was bereaved of her h...
  rank 2 ·: Volume 5, Book 59, Number 300:
Narrated Anas:
The Prophet said, "Who will go and...
  rank 3 ·: Volume 7, Book 63, Number 255:
Narrated Um 'Atiyya:
The Prophet said, "It is not...

Generating answer...
✓ routed to: openai/gpt-oss-120b:free
According to the Prophetic tradition it is **not permissible** for a widow to apply kohl after the death of her husband.

- **Volume 7, Book 63, Number 255** – The Prophet said: “It is not lawful for a lady who believes in Allah and the Last Day to mourn for more than three days for a dead person, except for her husband, **in which c

In [32]:
# Cell 13b — quick end-to-end test with OpenRouter
# proves: retrieve → prompt → LLM → answer works before building UI

import requests
import os
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY') # get free key at openrouter.ai
MODEL = "openrouter/free"  # free tier model

def rag_answer(query: str, collection, top_k=3) -> str:
    # step 1: retrieve with MMR
    result = mmr_retrieve(collection, query, k=top_k, fetch_k=25)
    chunks = result[0]
    sims   = result[1]
    context = "\n\n---\n\n".join(chunks)

    # step 2: build prompt
    prompt = f"""You are a knowledgeable assistant for Sahih Bukhari.
Use ONLY the provided hadith excerpts to answer the question.
If the answer is not in the excerpts, say "I could not find this in the provided hadiths."

Hadith excerpts:
{context}

Question: {query}

Answer:"""

    # step 3: LLM call
    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Content-Type": "application/json","Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": prompt}]
        }
    )


    
    return resp.json()["choices"][0]["message"]["content"]

# test it
answer = rag_answer(
    "Tell me how Islam Sees Saving Money? For Example: If my friends come to my house and I only have 10 dollars. There IS an option not to spend that money ie save and an option to spend it. What should I do in Islamic view?  ",
    collections["hadith_aware"]
)
print(answer)

I could not find this in the provided hadiths.
